# Práctica – Clase 6: Secuencias y redes recurrentes

Este cuaderno es la práctica que completas por tu cuenta tras la Clase 6 del curso *Advanced Deep Learning*. Aquí aplicas lo que viste sobre la red neuronal recurrente (RNN) y, en concreto, sobre la memoria a corto-largo plazo (LSTM, del inglés *Long Short-Term Memory*).

## Qué vas a practicar

Vas a construir y usar un generador de texto carácter a carácter con una LSTM en PyTorch:

- **Ejercicio 1**: completas la definición de una red LSTM que predice el siguiente carácter de un texto.
- **Ejercicio 2**: usas ese modelo entrenado para generar texto nuevo, carácter a carácter, a partir de una semilla.

## Requisito previo

Haber visto la Clase 6, "Secuencias y redes recurrentes", donde se explica el estado oculto, la LSTM y sus puertas. Si algo de esas ecuaciones no te resulta familiar, repasa el capítulo del manual antes de seguir.

## Cómo trabajar con este cuaderno

Cada ejercicio tiene el mismo formato: primero el enunciado, después una o varias celdas de código con partes marcadas como `# TODO`, que debes completar tú. Ejecuta las celdas en orden, de arriba hacia abajo: el ejercicio 2 depende del modelo que entrenas en el ejercicio 1.

Intenta resolver cada `TODO` sin mirar la solución. Debajo de cada ejercicio encuentras la solución colapsada, por si te atascas o quieres comparar tu enfoque.

## Ejercicio 1: define la LSTM del generador de texto

**Caso**: quieres construir un modelo capaz de predecir el siguiente carácter de un texto, dado el fragmento que lleva escrito hasta ese punto. Es la misma idea que el teclado de tu móvil cuando te sugiere la letra siguiente, pero aplicada carácter a carácter en lugar de palabra a palabra.

El corpus de entrenamiento es un texto corto, definido en la propia celda de código, sobre el trabajo de un hub de innovación digital. Se repite varias veces para que el modelo tenga suficientes ejemplos del mismo patrón y pueda converger en pocas épocas, algo habitual cuando entrenas con un corpus diminuto.

Ya tienes escrito todo el andamiaje de tokenización, vocabulario y bucle de entrenamiento. Tu tarea es completar la clase `ModeloCharLSTM`: la capa que convierte cada carácter en un vector, la capa recurrente y la capa que traduce el estado oculto en una predicción sobre el vocabulario.

**Pistas**:

- Un carácter llega al modelo como un índice entero. Para convertirlo en un vector denso, usa `nn.Embedding(tam_vocab, tam_embedding)`.
- La capa recurrente es `nn.LSTM(tam_embedding, tam_oculto, num_layers=num_capas, batch_first=True)`. El argumento `batch_first=True` hace que la forma de los tensores sea (lote, tiempo, características), en lugar de (tiempo, lote, características).
- `nn.LSTM` devuelve dos elementos: la salida en cada paso temporal y el estado final `(h, c)`. En el `forward`, propaga ambos: la salida hacia la capa lineal, y el estado hacia quien llamó al modelo (lo necesitarás en el ejercicio 2, para generar texto paso a paso sin reprocesar todo lo anterior).
- La capa final es una `nn.Linear(tam_oculto, tam_vocab)`: convierte el estado oculto de cada paso en un vector de logits, uno por cada carácter posible del vocabulario.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# Texto de entrenamiento: se repite para que la LSTM tenga suficientes
# ejemplos del mismo patron y pueda converger en pocas epocas
texto_entrenamiento = (
    "el hub europeo de innovacion digital ayuda a empresas del sector "
    "agroalimentario, forestal y de la salud a adoptar inteligencia artificial. "
    "las empresas que participan en el hub mejoran sus procesos con datos "
    "y con modelos de aprendizaje automatico. "
) * 30

# Vocabulario: cada caracter distinto del texto se convierte en un indice entero
caracteres = sorted(set(texto_entrenamiento))
char_a_idx = {c: i for i, c in enumerate(caracteres)}
idx_a_char = {i: c for i, c in enumerate(caracteres)}
tam_vocab = len(caracteres)

print(f"Tamano del vocabulario: {tam_vocab} caracteres distintos")

# Codificamos todo el texto como un tensor de indices enteros
datos = torch.tensor([char_a_idx[c] for c in texto_entrenamiento], dtype=torch.long)

longitud_secuencia = 40


def crear_lote(datos, longitud_secuencia, tam_lote):
    # Elegimos puntos de inicio aleatorios y recortamos ventanas de longitud fija;
    # el objetivo de cada ventana es la misma ventana desplazada un caracter
    indices_inicio = torch.randint(0, len(datos) - longitud_secuencia - 1, (tam_lote,))
    entradas = torch.stack([datos[i:i + longitud_secuencia] for i in indices_inicio])
    objetivos = torch.stack([datos[i + 1:i + longitud_secuencia + 1] for i in indices_inicio])
    return entradas, objetivos

In [ ]:
class ModeloCharLSTM(nn.Module):
    def __init__(self, tam_vocab, tam_embedding=16, tam_oculto=64, num_capas=1):
        super().__init__()

        # TODO: crea una capa nn.Embedding que convierta cada indice de caracter
        # (entero entre 0 y tam_vocab - 1) en un vector denso de tamano tam_embedding
        self.embedding = ...

        # TODO: crea una capa nn.LSTM que reciba vectores de tamano tam_embedding,
        # produzca un estado oculto de tamano tam_oculto, use num_capas capas
        # apiladas y trabaje con tensores en formato (lote, tiempo, caracteristicas)
        self.lstm = ...

        # TODO: crea una capa nn.Linear que transforme el estado oculto de la LSTM
        # (tamano tam_oculto) en un vector de logits sobre el vocabulario (tamano tam_vocab)
        self.salida = ...

    def forward(self, x, estado=None):
        # TODO: pasa x por la capa de embedding
        x = ...

        # TODO: pasa x (y el estado, si se recibio uno) por la LSTM.
        # nn.LSTM devuelve una tupla (salida_en_cada_paso, estado_final);
        # si estado es None, PyTorch lo inicializa en ceros automaticamente
        salida_lstm, estado = ...

        # TODO: pasa la salida de la LSTM por la capa lineal de salida
        # para obtener los logits sobre el vocabulario en cada paso temporal
        logits = ...

        return logits, estado

In [ ]:
modelo = ModeloCharLSTM(tam_vocab)
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.003)

num_epocas = 300
tam_lote = 32

for epoca in range(num_epocas):
    entradas, objetivos = crear_lote(datos, longitud_secuencia, tam_lote)
    logits, _ = modelo(entradas)
    # reorganizamos (lote, tiempo, vocab) en (lote*tiempo, vocab) para tratar
    # cada posicion de cada secuencia como un ejemplo de clasificacion independiente
    perdida = F.cross_entropy(logits.reshape(-1, tam_vocab), objetivos.reshape(-1))
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
    if (epoca + 1) % 50 == 0:
        print(f"Epoca {epoca + 1}/{num_epocas} - perdida: {perdida.item():.4f}")

<details>
<summary>💡 Ver solución</summary>

```python
class ModeloCharLSTM(nn.Module):
    def __init__(self, tam_vocab, tam_embedding=16, tam_oculto=64, num_capas=1):
        super().__init__()

        # Convierte cada indice de caracter en un vector denso de tamano tam_embedding
        self.embedding = nn.Embedding(tam_vocab, tam_embedding)

        # Capa recurrente: recibe vectores de tamano tam_embedding y produce
        # un estado oculto de tamano tam_oculto en cada paso temporal
        self.lstm = nn.LSTM(tam_embedding, tam_oculto, num_layers=num_capas, batch_first=True)

        # Traduce el estado oculto de cada paso en logits sobre el vocabulario
        self.salida = nn.Linear(tam_oculto, tam_vocab)

    def forward(self, x, estado=None):
        x = self.embedding(x)
        # estado es la tupla (h, c); si se pasa None, PyTorch la inicializa en ceros
        salida_lstm, estado = self.lstm(x, estado)
        logits = self.salida(salida_lstm)
        return logits, estado
```
</details>

## Ejercicio 2: genera texto nuevo con el modelo entrenado

**Caso**: ya tienes un modelo entrenado que predice, a partir de un fragmento de texto, la distribución de probabilidad del carácter siguiente. Ahora quieres usarlo para generar texto nuevo, carácter a carácter, a partir de una semilla que tú eliges (por ejemplo, `"el hub "`).

La idea es sencilla: le das al modelo la semilla, obtienes el estado de la LSTM tras procesarla, y a partir de ahí repites, `n_caracteres` veces, el mismo ciclo: predices el carácter siguiente, lo añades al texto generado y lo usas como entrada del siguiente paso.

Completa la función `generar_texto(modelo, texto_inicial, n_caracteres)`. El andamiaje ya prepara el estado inicial a partir de la semilla; tu tarea es completar el bucle que predice y añade un carácter en cada iteración.

**Pistas**:

- Dentro del bucle, llama al modelo con `ultimo_caracter` (el último carácter generado, no todo el texto) y con `estado`. Como el modelo devuelve el estado actualizado, no necesitas reprocesar el texto completo en cada paso: por eso pasar el estado es tan útil en generación.
- Los logits que te interesan son los del último paso temporal: `logits[:, -1, :]`. Conviértelos en probabilidades con `F.softmax(..., dim=-1)`, dividiendo antes por `temperatura` para controlar cuánto se afila o suaviza la distribución.
- Para elegir el siguiente carácter no uses siempre el más probable: usa `torch.multinomial(probabilidades, num_samples=1)`, que muestrea según esas probabilidades y da variedad al texto generado.
- Traduce el índice muestreado a carácter con el diccionario `idx_a_char`, que ya tienes construido en el ejercicio 1.

In [ ]:
@torch.no_grad()
def generar_texto(modelo, texto_inicial, n_caracteres, temperatura=0.8):
    modelo.eval()

    entrada = torch.tensor([[char_a_idx[c] for c in texto_inicial]], dtype=torch.long)
    _, estado = modelo(entrada)  # procesamos la semilla para construir el estado inicial
    ultimo_caracter = entrada[:, -1:]
    generado = texto_inicial

    for _ in range(n_caracteres):
        # TODO: pasa ultimo_caracter y estado por el modelo para obtener los
        # logits del siguiente caracter y el estado actualizado de la LSTM
        logits, estado = ...

        # TODO: aplica softmax con temperatura sobre los logits del ultimo paso
        # (logits[:, -1, :] dividido entre temperatura) para obtener una
        # distribucion de probabilidad sobre el vocabulario
        probabilidades = ...

        # TODO: muestrea un indice de caracter a partir de probabilidades
        # usando torch.multinomial con num_samples=1
        idx_siguiente = ...

        generado += idx_a_char[idx_siguiente.item()]
        ultimo_caracter = idx_siguiente

    modelo.train()
    return generado


print(generar_texto(modelo, "el hub ", 120))

<details>
<summary>💡 Ver solución</summary>

```python
@torch.no_grad()
def generar_texto(modelo, texto_inicial, n_caracteres, temperatura=0.8):
    modelo.eval()

    entrada = torch.tensor([[char_a_idx[c] for c in texto_inicial]], dtype=torch.long)
    _, estado = modelo(entrada)  # procesamos la semilla para construir el estado inicial
    ultimo_caracter = entrada[:, -1:]
    generado = texto_inicial

    for _ in range(n_caracteres):
        logits, estado = modelo(ultimo_caracter, estado)
        # la temperatura suaviza (>1) o afila (<1) la distribucion antes de muestrear
        probabilidades = F.softmax(logits[:, -1, :] / temperatura, dim=-1)
        idx_siguiente = torch.multinomial(probabilidades, num_samples=1)
        generado += idx_a_char[idx_siguiente.item()]
        ultimo_caracter = idx_siguiente

    modelo.train()
    return generado


print(generar_texto(modelo, "el hub ", 120))
```
</details>

## Para terminar

Acabas de construir, entrada por entrada, el mecanismo que convierte una LSTM en un generador de texto: el estado oculto que resume lo leído hasta ahora, y el bucle que reutiliza ese estado para predecir un carácter tras otro sin volver a procesar todo el texto desde el principio. Es la misma lógica que sostiene sistemas mucho más grandes, desde el autocompletado de tu móvil hasta los modelos de lenguaje actuales.

Prueba a cambiar la semilla, la temperatura o el propio texto de entrenamiento, y observa cómo cambia el resultado: es la forma más rápida de desarrollar intuición sobre lo que controla cada pieza del modelo.

**Guarda tu cuaderno con los resultados y compáralos con el resto de la clase antes de la próxima sesión.**